In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

In [2]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

In [4]:
import pandas as pd

BASE = "Cantill_DataML_Task_Kit"
kw = dict(dtype=str, keep_default_na=False)    

erp = pd.read_csv(f"{BASE}/company_A/erp_goods_receipts.csv", **kw)
tool = pd.read_csv(f"{BASE}/company_A/invoice_tool_export.csv", **kw)
labA = pd.read_csv(f"{BASE}/company_A/labels_A.csv", **kw)
taxA = pd.read_csv(f"{BASE}/company_A/taxonomy_A.csv", **kw)

b = pd.read_csv(f"{BASE}/company_B/invoice_lines.csv", **kw)
seed = pd.read_csv(f"{BASE}/company_B/seed_labels.csv", **kw)
fx = pd.read_csv(f"{BASE}/company_B/fx_rates_usd.csv", **kw)
taxB = pd.read_csv(f"{BASE}/company_B/taxonomy_B.csv", **kw)
 
print(erp.shape, erp.dtypes.value_counts().to_dict())    
print(erp["posting_date"].eq("").sum())                  

(37029, 26) {<StringDtype(storage='python', na_value=nan)>: 26}
24370


In [5]:
for n, d in dict(erp=erp, tool=tool, labA=labA, taxA=taxA, b=b, seed=seed, fx=fx, taxB=taxB).items():
    print(n, d.shape)

erp (37029, 26)
tool (28520, 11)
labA (37029, 6)
taxA (40, 3)
b (41028, 28)
seed (3000, 6)
fx (683, 3)
taxB (40, 3)


In [6]:
def profile(df):
    s = df.astype(str)
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nunique": df.nunique(),
        "blank_%": (s.apply(lambda c: c.str.strip().eq("") | c.eq("nan")).mean() * 100).round(1),
        "example": df.iloc[0],
    })

In [7]:
profile(erp)

,dtype,nunique,blank_%,example
line_id,str,37029,0.0,A000002
source_system,str,4,0.0,ERP-A1
spend_type,str,4,0.0,Material
po_number,str,20670,0.0,4510000018
po_item,str,74,0.0,10.0
receipt_document,str,31522,0.0,5010000009
receipt_item,str,64,0.0,1.0
posting_date,str,887,65.8,2021-06-28
material_number,str,3140,42.7,M1000001
material_short_text,str,4336,0.0,"FAN ELEC;22KW,YARVEX FLW,DIA 760MM,FLNG"


In [8]:
for c in ["source_system", "spend_type", "material_type", "document_currency", "opex_capex", "contract_status"]:
    print(erp[c].value_counts(dropna=False).to_string(), "\n")

source_system
ERP-A1    36908
ERP-A2      107
ERP-A3        9
ERP-A4        5 

spend_type
Material     17959
Service      11232
Free Text     5551
Catalogue     2287 

material_type
                           19070
Consumables                 9081
Spare parts                 5758
Operations Material         2028
Refurbishment Material       777
Non-stock materials          302
HIBE_Operating supplies        6
Spare Parts                    4
Trading Goods                  2
Waste                          1 

document_currency
ZAR    21197
       15816
EUR       13
CHF        2
MZN        1 

opex_capex
Opex     35386
Capex     1643 

contract_status
On Contract     23829
Non Contract    13200 



In [9]:
erp["no_date"] = erp["posting_date"].eq("")
print(pd.crosstab(erp["no_date"], erp["spend_type"]))
print(pd.crosstab(erp["no_date"], erp["source_system"]))

blank = erp.drop(columns="no_date").eq("")
print(blank.groupby(erp["no_date"]).mean().T.round(2))

erp[erp["no_date"]].sample(8, random_state=1)

spend_type  Catalogue  Free Text  Material  Service
no_date                                            
False            1380       1287      9405      587
True              907       4264      8554    10645
source_system  ERP-A1  ERP-A2  ERP-A3  ERP-A4
no_date                                      
False           12623      25       6       5
True            24285      82       3       0
no_date              False  True 
line_id               0.00   0.00
source_system         0.00   0.00
spend_type            0.00   0.00
po_number             0.00   0.00
po_item               0.00   0.00
receipt_document      0.00   0.00
receipt_item          0.00   0.00
posting_date          0.00   1.00
material_number       0.00   0.65
material_short_text   0.00   0.00
service_short_text    0.95   1.00
material_type         0.26   0.65
quantity              0.00   0.00
uom                   0.00   1.00
po_uom                0.00   1.00
value_zar             0.00   0.00
unit_cost_zar         0.00   1

,line_id,source_system,spend_type,po_number,po_item,receipt_document,receipt_item,posting_date,material_number,material_short_text,service_short_text,material_type,quantity,uom,po_uom,value_zar,unit_cost_zar,document_currency,vendor_id,supplier_name,site_code,site_name,cost_centre,opex_capex,contract_status,contract_number,no_date
15150,A018554,ERP-A1,Service,4510049729,10.0,5010085721,2.0,,,"MNTN,BELT,CNVY;INST/RECLM,NT",,,180.0,,,21234.96,,,,FALRIDGE MINING CONSTRUCTION SERVICES,,Site 02,,Opex,On Contract,CW100411,True
15530,A018993,ERP-A1,Service,4510050774,10.0,5010087173,5.0,,,"MNTN,BELT,CNVY;SD WAL&ROF,BARNG,/TEAM",,,14.0,,,554631.65,,,,YARBURY SA (PTY) LTD,,Site 04,,Opex,On Contract,CW100486,True
30252,A036753,ERP-A1,Material,4510098721,10.0,5010159140,1.0,,M1008918,"O RING;P/N:00910660-0363,ID:6-1/2IN,BLK",,Non-stock materials,4.0,,,94.66,,ZAR,,PRAHAVEN (PTY) LTD,,Site 07,,Opex,On Contract,CW100298,True
20512,A024982,ERP-A1,Service,4510067430,10.0,5010112028,3.0,,,"ANLYZ,SMPL;CLRIMTRC 65/80,UG SGMA,STNE",,,1027.0,,,77108.19,,,,PRASTONE SOLUTIONS SUPPLIES,,Site 06,,Opex,On Contract,CW100260,True
17155,A020946,ERP-A1,Material,4510056091,20.0,5010095341,1.0,,M1010900,"CARRIER;P/N:100039130,BRNG",,Spare parts,2.0,,,27632.4,,ZAR,,PRAHAVEN (PTY) LTD,,Site 07,,Opex,On Contract,CW100274,True
29495,A035796,ERP-A1,Free Text,4510095580,10.0,5010155114,1.0,,,VehicleMaintRepairSV,,,1.0,,,14355.0,,,,ROSWICK CC,,Site 06,,Opex,Non Contract,,True
20146,A024538,ERP-A1,Service,4510066191,10.0,5010110334,2.0,,,"INST,SPRT;ROFBLT,NT,2.4M",,,100.0,,,26785.54,,,,PELWICK (PTY) LTD,,Site 02,,Opex,On Contract,CW100364,True
19696,A024023,ERP-A1,Free Text,4510064906,20.0,5010108283,1.0,,,Couplings,,,4.0,,,916.11,,,,CALWOOD (PTY) LTD,,Site 02,,Opex,Non Contract,,True


In [ ]:
erp["v"] = erp.value_zar.astype(float)
print(erp.v.describe())
print("negative:", (erp.v < 0).sum(), " zero:", (erp.v == 0).sum())
print(erp.sort_values("v").head(8)[["line_id", "po_number", "material_short_text", "v", "supplier_name"]])
print(erp.sort_values("v").tail(8)[["line_id", "po_number", "material_short_text", "v", "supplier_name"]])

neg = erp[erp.v < 0].assign(absv=lambda d: d.v.abs().round(2))
pos = erp[erp.v > 0].assign(absv=lambda d: d.v.round(2))
print("negatives with a same-PO, same-abs-value positive:",
      neg.merge(pos, on=["po_number", "absv"]).line_id_x.nunique(), "of", len(neg))

count    3.702900e+04
mean     6.895198e+04
std      2.634627e+05
min     -2.050368e+07
25%      4.781890e+03
50%      1.684356e+04
75%      5.308862e+04
max      1.596788e+07
Name: v, dtype: float64
negative: 434  zero: 0
       line_id   po_number                       material_short_text            v                 supplier_name
5574   A006809  4510014625                            POWDER;STNEDST -20503682.72              NEXORA (PTY) LTD
10003  A012252  4510033733   PICK, DRUM CUTTER;P/N:1924847,SHNK:30MM  -2736115.20               DUNAX (PTY) LTD
1770   A002178  4510006345   PICK, DRUM CUTTER;P/N:3113108,SHNK:30MM  -1933576.74               DUNAX (PTY) LTD
28261  A034305  4510091597                   BLD,WAL,UG;AIR XNG,SNGL  -1272386.57              KINLEY (PTY) LTD
8154   A010035  4510027567   ROD DRIL;P/N:5950087,CONCTN:M16 DIN 405   -917625.28               DUNAX (PTY) LTD
5811   A007104  4510019328  MESH;4,8M,WD:1,2M,WD 75 X LG 100MM,DIA 4   -731817.90              PRATON (PT

In [ ]:
erp["v"] = erp["value_zar"].astype(float)       
chk = erp[~erp["no_date"]].copy()
chk["q"] = pd.to_numeric(chk["quantity"], errors="coerce")
chk["uc"] = pd.to_numeric(chk["unit_cost_zar"], errors="coerce")
chk["diff"] = (chk["q"] * chk["uc"] - chk["v"]).abs() / chk["v"].abs().clip(lower=1)
print((chk["diff"] < 0.01).mean(), "of dated rows satisfy qty x unit_cost = value")
chk[chk["diff"] > 0.01].head(8)[["line_id", "quantity", "unit_cost_zar", "value_zar", "uom", "po_uom"]]

0.9974721541985939 of dated rows satisfy qty x unit_cost = value


,line_id,quantity,unit_cost_zar,value_zar,uom,po_uom
1731,A002132,4.0,8.32,8317.2,PAK,EA
1804,A002216,4.0,8.32,8317.2,PAK,EA
1943,A002393,80.0,0.44,34.8,EA,EA
2094,A002579,1852273.09,0.87,191400.0,LE,LE
2095,A002580,2379224.0,0.87,450976.68,LE,LE
2181,A002688,2.0,8.32,4158.6,PAK,EA
2252,A002772,4.0,8.32,8317.2,PAK,EA
2480,A003051,8.0,8.32,16634.4,PAK,EA


In [13]:
erp["record_id"] = "ERP:" + erp.line_id
e = erp.merge(labA, on="record_id", how="left")
print("unlabelled ERP rows:", e.category_l1.isna().sum())

print(e.category_l1.value_counts(), "\n")
print(e.groupby(["category_l1", "category_l2", "category_l3"]).agg(n=("v", "size"), spend=("v", "sum"))
        .sort_values("spend", ascending=False).round(0).to_string())
print("\nrows where L2 == L3:", (e.category_l2 == e.category_l3).mean().round(3))

unlabelled ERP rows: 0
category_l1
Mining Operating Materials                  13166
Underground Production Equipment             8634
Underground Services                         6350
Flameproof Diesel Equipment                  5782
Mining Support Equipment and Consumables     3097
Name: count, dtype: int64 

                                                                                                                                         n        spend
category_l1                              category_l2                                   category_l3                                                     
Mining Operating Materials               Roof Bolts                                    Roof Bolts                                     1965  505463921.0
                                         Picks and Sleeves                             Picks and Sleeves                              1131  250387632.0
Underground Services                     Conveyors and Stone Dust Barrier Servi

In [14]:
print(pd.crosstab(e.spend_type, e.category_l1, normalize="index").round(2))

mm = e[e.material_number != ""].groupby("material_number").category_l3.nunique()
print("material_numbers with >1 L3:", (mm > 1).sum(), "of", len(mm))

category_l1  Flameproof Diesel Equipment  Mining Operating Materials  Mining Support Equipment and Consumables  Underground Production Equipment  Underground Services
spend_type                                                                                                                                                            
Catalogue                           0.00                        1.00                                      0.00                              0.00                  0.00
Free Text                           0.41                        0.17                                      0.09                              0.22                  0.11
Material                            0.10                        0.54                                      0.12                              0.24                  0.00
Service                             0.15                        0.02                                      0.04                              0.28                  0.5

In [15]:
g = e.groupby("supplier_key").supplier_name.agg(lambda s: sorted(s.unique()))
print("names per golden supplier:\n", g.map(len).describe())
for k, names in g.sort_values(key=lambda s: s.map(len), ascending=False).head(6).items():
    print(k, names, "\n")
 
v = e[e.vendor_id != ""]
print("vendor_ids per golden key:", v.groupby("supplier_key").vendor_id.nunique().describe().round(1).to_dict())
print("golden keys per vendor_id:", v.groupby("vendor_id").supplier_key.nunique().value_counts().to_dict())

names per golden supplier:
 count     70.000000
mean       2.757143
std       14.459775
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max      122.000000
Name: supplier_name, dtype: float64
SG-f85711 ['ALTON CONSTRUCTION LABOUR SERVICES (PTY) LTD', 'ASHARI (PTY) LTD', 'ASHDEN (PTY) LTD', 'ASHMONT (PTY) LTD', 'ASHVALE SERVICES', 'ASHWICK ELECTRICAL (PTY) LTD', 'BLYARI HYDRAULICS (PTY) LTD', 'BLYCREST FILTERS (PTY) LTD', 'BLYMORE HYDRAULICS (PTY) LTD', 'BLYVALE MINING CONSTRUCTION SERVICES CC', 'BRETON PRODUCTS', 'CALBURY ENGINEERING SOLUTIONS (PTY) LTD', 'CALFIELD EQUIPMENT', 'CALRIDGE MOTOR (PTY) LTD', 'CALWOOD (PTY) LTD', 'CORORA', 'CORSTON MINING (PTY) LTD', 'DALARI PIPELINE EQUIPMENT BV', 'DALLEY (PTY) LTD', 'DALMONT CC', 'DUNBROOK STEEL', 'ELDLEY (PTY) LTD', 'ELDTON (PTY) LTD', 'ESTAX ELECTRICAL INDUSTRIAL (PTY) LTD', 'ESTION MOTOR ENGINEERING (PTY) LTD', 'ESTSTONE (PTY) LTD', 'ESTVALE ELECTRICAL (PTY) LTD', 'FALICA (PTY) LTD', 'FALORA (PTY) LTD', 

In [16]:
print(pd.crosstab(e.material_number.eq(""), e.item_key.eq(""), rownames=["no material#"], colnames=["no item_key"]))

ik = e[e.item_key != ""]
print("material_numbers per item_key:", ik.groupby("item_key").material_number.nunique().value_counts().head(6).to_dict())
print("distinct texts per item_key:", ik.groupby("item_key").material_short_text.nunique().value_counts().head(6).to_dict())
 
multi = ik.groupby("item_key").material_short_text.nunique()
for k in multi[multi > 1].index[:5]:
    print(k, ik[ik.item_key == k].material_short_text.unique()[:6])

no item_key   False  True 
no material#              
False         21213      0
True              0  15816
material_numbers per item_key: {1: 2406, 4: 3, 64: 2, 2: 2, 192: 1, 45: 1}
distinct texts per item_key: {1: 2386, 2: 37, 3: 2, 794: 1}
IG-06a8b34 <StringArray>
['SEAMEX WATER EXPLOSIVE GEL MERICA', 'SUG: SUPPLY SUB COOLERS 12000BT P/N: ACS', 'SUG: SUPPLY SUB COOLERS 32000BT P/N: ACS', '(Bosjespruit South Irenedale) STONE, CRU', 'Resin slow 5-10min yellow', 'Irenedale South  Stone 37,5mm']
Length: 6, dtype: str
IG-12cff7d <StringArray>
['DETONATOR, EXPLOSIVES;3,5M,7', 'DETONATOR, EXPLOSIVES;3.5M,7']
Length: 2, dtype: str
IG-1c65ac7 <StringArray>
['TRNSP RD,PROD,HAZ;EXPL&DTNTR,100-140 CSE', 'TRNSP RD,PROD,HAZ;EXPL&DTNTR,300 CSE', 'TRNSP RD,PROD,HAZ;EXPL&DTNTR,16 CSE']
Length: 3, dtype: str
IG-1ca03f9 <StringArray>
['ROD DRIL;DIA 60 MM X LG 2,095 M,STL,RED', 'ROD DRIL;DIA 60 MM X LG 2.095 M,STL,RED']
Length: 2, dtype: str
IG-1ecb19c <StringArray>
['BIT RCK;P/N:6189499,TWO PRONG,DIA:

In [17]:
e["text"] = np.where(e.service_short_text != "", e.service_short_text, e.material_short_text)
print(e.text.str.len().describe())
for l1, g in e.groupby("category_l1"):
    print("\n==", l1)
    print(g.text.sample(6, random_state=0).to_string(index=False))

count    37029.000000
mean        28.990575
std          9.347552
min          3.000000
25%         21.000000
50%         31.000000
75%         38.000000
max         44.000000
Name: text, dtype: float64

== Flameproof Diesel Equipment
                      LORDEN MACHINAR
                 Mechanical Maintenan
COUPLING, PIPE HALF;P/N:A2U900-522024
              GSKT PRE CUT;P/N:4P7704
REPR;GENERATOR, ENGINE ACCESSORY;700W
                 VehicleMaintRepairSV

== Mining Operating Materials
RESIN, ROOF BOLT;1500-1800MM,6S,23MM,900
ROD ANCHR;DIA:20MM,1800MM,REB QC(HR),ROF
   STICK;SOUNDING,DIA 50 X LG 3500MM,WOD
      MESH;WLD,6M,WD:2,2M,100 X 75MM,4MM
                          POWDER;STNEDST
                          POWDER;STNEDST

== Mining Support Equipment and Consumables
    PIN, SHEAR;P/N:30-8526_W50_B_WY,BRS
        SPACER;P/N:1544706-06,LABYRINTH
FAN ELEC;22KW,YARVEX FLW,DIA 760MM,FLNG
          SHAFT ASSY;P/N:52-1440,HD,STL
            SEGMENT;SECONDARY CRSHR,STL
          Ratch

In [18]:
def shape_of(s):
    """Replace digits with 9 and letters with a, to reveal formats."""
    return re.sub(r"[A-Za-z]", "a", re.sub(r"\d", "9", s))

In [19]:
print(tool.status.value_counts(), "\n")
print(tool.vat_rate.value_counts(), "\n")
for c in ["invoice_date", "po_reference", "gross_amount"]:
    print(c, tool[c].map(shape_of).value_counts().head(10).to_dict(), "\n")
print(tool.currency.value_counts())

status
Paid         19837
Approved      8180
Cancelled      188
On Hold        172
Rejected       143
Name: count, dtype: int64 

vat_rate
15      6920
0.15    6826
15%     6652
        6542
0       1580
Name: count, dtype: int64 

invoice_date {'': 18660, '99/99/9999': 3362, '9999-99-99': 3283, '99-aaa-99': 3215} 

po_reference {'9999999999': 5895, 'aa#9999999999': 5820, 'aa 9999999999': 5807, '9999999999/99.9': 5705, '': 5199, '9999999999/999.9': 94} 

gross_amount {'99999.99': 8767, '9999.99': 5901, '99,999.99': 3747, '999999.99': 3347, '9,999.99': 2596, '999,999.99': 1421, '999.99': 1401, '9.99': 483, '99.99': 308, '9999999.99': 159} 

currency
ZAR    28520
Name: count, dtype: int64


In [20]:
def parse_vat(s):
    s = s.strip().replace("%", "")
    if s == "": return 0.15                        
    v = float(s)
    return v / 100 if v > 1 else v

def parse_date(s):
    s = s.strip()
    if not s: return pd.NaT
    for f in ("%Y-%m-%d", "%d-%b-%y", "%d/%m/%Y", "%m/%d/%Y"):
        try: return pd.to_datetime(s, format=f)
        except ValueError: pass
    return pd.NaT

t = tool.copy()
t["gross"] = t.gross_amount.str.replace(",", "").astype(float)
t["vat"]   = t.vat_rate.map(parse_vat)
t["net"]   = t.gross / (1 + t.vat)
t["date"]  = t.invoice_date.map(parse_date)
t["po"]    = t.po_reference.str.extract(r"(45\d{8})")[0]
t["po_item"] = t.po_reference.str.extract(r"/(\d+)\.\d")[0]

print("unparsed non-blank dates:", (t.date.isna() & t.invoice_date.str.strip().ne("")).sum())
# is dd/mm vs mm/dd ambiguous?
sl = t.invoice_date[t.invoice_date.str.contains("/")].str.split("/", expand=True).astype(int)
print("slash dates: first part >12:", (sl[0] > 12).sum(), " second part >12:", (sl[1] > 12).sum())
print(t.date.describe())
print("net by status:\n", t.groupby("status").net.sum().round(0))

unparsed non-blank dates: 0
slash dates: first part >12: 2050  second part >12: 0
count                          9860
mean     2023-01-14 17:40:43.326572
min             2021-06-28 00:00:00
25%             2022-03-24 18:00:00
50%             2023-01-09 00:00:00
75%             2023-11-13 00:00:00
max             2024-07-20 00:00:00
Name: date, dtype: object
net by status:
 status
Approved     5.880149e+08
Cancelled    9.993584e+06
On Hold      1.086211e+07
Paid         1.362694e+09
Rejected     6.666815e+06
Name: net, dtype: float64


In [21]:
dup_cols = ["invoice_number", "line_text", "gross_amount"]
d = t[t.duplicated(dup_cols, keep=False)].sort_values(dup_cols)
print("rows in duplicate groups:", len(d), " extra copies:", t.duplicated(dup_cols).sum())
print(d.head(10)[["tool_record_id"] + dup_cols + ["po_reference", "status", "invoice_date"]])
# are the duplicates exact copies, or do other fields differ?
print(d.groupby(dup_cols).agg(n=("tool_record_id", "size"), statuses=("status", "nunique"),
      pos=("po_reference", "nunique"), dates=("invoice_date", "nunique")).describe())

# same invoice number reused with different content (not a duplicate)
print("invoice numbers with >1 distinct supplier:", t.groupby("invoice_number").supplier_name_as_captured.nunique().gt(1).sum())

rows in duplicate groups: 1028  extra copies: 562
      tool_record_id invoice_number   line_text gross_amount      po_reference status invoice_date
11349       TA023739         105035  HAND TOOLS     1,217.83        4510100639   Paid             
19729       TA023737         105035  HAND TOOLS     1,217.83        4510100639   Paid             
13513       TA023746         105035  HAND TOOLS       243.42     PO#4510100639   Paid             
18086       TA023755         105035  HAND TOOLS       243.42  4510100639/360.0   Paid             
5707        TA023745         105035  HAND TOOLS       426.13     PO#4510100639   Paid             
13518       TA023750         105035  HAND TOOLS       426.13     PO#4510100639   Paid             
3880        TA023757         105035  HAND TOOLS       427.87        4510100639   Paid             
16117       TA023731         105035  HAND TOOLS       427.87  4510100639/510.0   Paid             
16714       TA023758         105035  Hand tools       365.2

In [22]:
names = t.supplier_name_as_captured
print(names.nunique(), "raw spellings")
norm = (names.str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
             .str.replace(r"\b(PTY|LTD|CC|SA|THE)\b", " ", regex=True)
             .str.split().str.join(" "))
print(norm.nunique(), "after simple normalisation vs", erp.supplier_name.nunique(), "ERP spellings")
print(names.value_counts().head(25))

702 raw spellings
315 after simple normalisation vs 193 ERP spellings
supplier_name_as_captured
PRAHAVEN (PTY) LTD                              1990
KELMONT (PTY) LTD                               1420
OAKENT MINING SA (PTY) LTD                      1123
DUNAX (PTY) LTD                                 1005
YARAX TECHNOLOGIES (PTY) LTD                     712
PELWICK (PTY) LTD                                590
LORDEN SOLUTIONS (PTY) LTD                       536
WEXTON ENGINEERING (PTY) LTD                     460
MERICA (PTY) LTD                                 460
DUNVALE MINING                                   429
ESTION MOTOR ENGINEERING (PTY) LTD               423
VERRICK                                          404
PELENT ENGINEERING (PTY) LTD                     370
KINMERE SA (PTY) LTD                             367
DUNTON (PTY) LTD                                 334
DALFIELD                                         331
VELDEN                                           324
YAR

In [23]:
erp["po"] = erp.po_number
erp_po = erp.groupby("po").v.agg(["sum", "size"])
m = t.dropna(subset=["po"]).join(erp_po, on="po", how="left")
print("tool lines with PO:", len(m), " PO found in ERP:", m["sum"].notna().sum())

# line-level: PO + net amount within tolerance
e_lines = erp[["line_id", "po", "po_item", "v", "quantity", "material_short_text", "service_short_text"]]
cand = m.merge(e_lines, on="po", how="inner", suffixes=("", "_erp"))
cand["err"] = (cand.net - cand.v).abs() / cand.v.abs().clip(lower=1)
best = cand.sort_values("err").drop_duplicates("tool_record_id")
for tol in (0.001, 0.01, 0.05, 0.2):
    print(f"tool lines matching an ERP line on PO and amount within {tol:.1%}:", (best.err < tol).sum())

tool lines with PO: 23321  PO found in ERP: 17620
tool lines matching an ERP line on PO and amount within 0.1%: 16532
tool lines matching an ERP line on PO and amount within 1.0%: 16881
tool lines matching an ERP line on PO and amount within 5.0%: 17050
tool lines matching an ERP line on PO and amount within 20.0%: 17180


In [24]:
near = best[(best.err >= 0.01) & (best.err < 0.5)]
print(len(near), "near-misses")
near.head(10)[["tool_record_id", "po", "net", "v", "err", "line_text", "material_short_text", "quantity", "quantity_erp"]]

473 near-misses


,tool_record_id,po,net,v,err,line_text,material_short_text,quantity,quantity_erp
68828,TA001988,4510006944,4631.026087,4584.81,0.010080,the Company Ballast Stone 53mm,the Company Ballast Stone 53mm,20.04,19.84
29668,TA003144,4510012316,20259.452174,20467.05,0.010143,POWDER;STNEDST,POWDER;STNEDST,33.18,33.52
110430,TA000133,4510000803,23323.339130,23088.87,0.010155,POWDER;STNEDST,POWDER;STNEDST,33.82,33.48
116810,TA007860,4510032756,24582.913043,24838.24,0.010280,P0WDER;STNEDST,POWDER;STNEDST,34.66,35.02
24647,TA001996,4510006944,4071.791304,4030.20,0.010320,THE COMPANY BALLAST STONE 53MM,the Company Ballast Stone 53mm,17.62,17.44
72495,TA003410,4510014724,5361.339130,5417.56,0.010378,"IRENEDALE SOUTH STONE 37,5MM","Irenedale South Stone 37,5mm",20.98,21.2
6624,TA001034,4510004892,6165.460870,6230.17,0.010386,the Company Ballast 5tone 53mm,the Company Ballast Stone 53mm,26.68,26.96
22544,TA009482,4510039346,4524.652174,4478.01,0.010416,"the Company Concrete Stone 37,5mm","the Company Concrete Stone 37,5mm",19.4,19.2
117966,TA001493,4510006820,5734.426087,5673.10,0.010810,"Ireneda1e South Stone 37,5mm","Irenedale South Stone 37,5mm",22.44,22.2
3638,TA001496,4510006820,5734.426087,5673.10,0.010810,"1reedaleS0uth Stone 37,5mm","Irenedale South Stone 37,5mm",22.44,22.2


In [25]:
# several ERP lines for one PO may sum to one tool invoice line
sums = t.dropna(subset=["po"]).groupby("po").net.sum().to_frame("tool_net").join(erp_po.rename(columns={"sum": "erp_net"}), how="inner")
sums["err"] = (sums.tool_net - sums.erp_net).abs() / sums.erp_net.abs().clip(lower=1)
print("POs where total tool net is within 1% of total ERP:", (sums.err < 0.01).mean().round(3))

POs where total tool net is within 1% of total ERP: 0.796


In [26]:
no_po = t[t.po.isna()]
unmatched_po = t[t.po.notna() & ~t.po.isin(erp.po)]
print("no PO:", len(no_po), " PO not in ERP:", len(unmatched_po))
print(no_po.sample(10, random_state=0)[["tool_record_id", "supplier_name_as_captured", "line_text", "net", "status"]])

# can they be matched by supplier + amount + text instead?
erp_key = set(zip(erp.v.round(0)))
print("no-PO lines whose rounded net equals some ERP value:", no_po.net.round(0).isin(erp.v.round(0)).mean().round(3))
print("PO-less lines whose text appears in ERP:", no_po.line_text.str.upper().isin(set(erp.material_short_text.str.upper())).mean().round(3))

no PO: 5199  PO not in ERP: 5701
      tool_record_id      supplier_name_as_captured                                line_text            net    status
24745       TA008253              KELMONT (PTY) LTD         thc Company  8a1last Stone  53mm    4489.800000      Paid
11121       TA000605                       GLENMERE        ANCHOR;PN:VE 20011127/00 CB,400MM   20384.356522  Approved
26900       TA002975   YARAX TECHNOLOGIES (PTY) LTD  ROD ANCHR;DIA:20MM,1500MM,REB QC(HR,ROF  201058.739130      Paid
2129        TA009219              Kelmont (Pty) Ltd         the Company  Ballast Stone  53mm    3476.886957      Paid
24474       TA026943           YARBURY SA (PTY) LTD           MNTN,BELT,CNV;4SECT,OPRN,/SECT  240908.070000      Paid
21650       TA001683                         VELDEN      REPR;DRUM;P/N:90-CA-UF15BA-HC,CRUSH  219792.380000  Approved
25450       TA007805                DUNVALE  MINING       ME5H;WLD,3M,WD:2,2M,100 X 75MM,4MM   77986.800000      Paid
15539       TA023771   

In [27]:
kw = r"TAX|VAT|INTERCOMPANY|ROYALT|INTEREST|BANK|LEVY|DIVIDEND|MANAGEMENT FEE|RECHARGE"
for name, df, col in [("tool", t, "line_text"), ("erp", erp, "material_short_text"), ("erp svc", erp, "service_short_text")]:
    hit = df[df[col].str.upper().str.contains(kw)]
    print(name, len(hit)); print(hit[col].value_counts().head(10).to_string(), "\n")
print(erp.supplier_name.str.upper().str.contains("GROUP|INTERNATIONAL|HOLDINGS").sum(), "ERP lines from group-like suppliers")

tool 14
line_text
CYLDR ASSY ACTNG LINEAR;ELEVATING FFT       4
REPR;VALVE;P/N:JHF14852600,BANK; BOOM,LH    2
REPR;VALVE;P/N:JHF14852600,BANK;OOM,LH      1
REPR;VALVE;P/N:JHF14852700,BANK;B0OM,RH     1
HANDLE;P/N:JHF4159700,VLV BANK,STL,BLK      1
CYLDR A5SY ACTNG LINEAR;ELEVATING FFT       1
REPR;VALVE;P/N:JHF14852700,BANK; BOOM,RH    1
CYLDRASSY ACTNG LINEAR;ELEVATING FFT        1
REPR;VALVE;P/N:JHF1852700,BANK;BOOM,RH      1
VALVE;P/N:JHF14852700,BANK; BOM 4.9         1 

erp 35
material_short_text
CYLDR ASSY ACTNG LINEAR;ELEVATING FFT       18
REPR;VALVE;P/N:JHF14852700,BANK; BOOM,RH     6
REPR;VALVE;P/N:JHF14852600,BANK; BOOM,LH     6
HANDLE;P/N:JHF4159700,VLV BANK,STL,BLK       4
VALVE;P/N:JHF14852700,BANK; BOM 4.9          1 

erp svc 0
Series([], ) 

0 ERP lines from group-like suppliers


In [28]:
display(profile(b))
for c in ["source_system", "region", "currency", "document_type", "po_type", "transport_flag"]:
    print(b[c].value_counts().to_string(), "\n")
print(pd.crosstab(b.source_system, b.region))

,dtype,nunique,blank_%,example
line_id,str,41028,0.0,B000001
source_system,str,8,0.0,ERP-B1 (JDE)
region,str,8,0.0,R1
invoice_number,str,17319,0.0,B-INV-1000001
invoice_line,str,493,0.0,001
posting_date,str,365,0.0,2024-03-29
invoice_date,str,590,0.0,2018-07-23
due_date,str,736,0.0,2024-03-29
paid_date,str,319,25.2,
line_description,str,12449,40.7,


source_system
ERP-B4 (JDE)    11000
ERP-B7 (SAP)    11000
ERP-B8 (SAP)     5003
ERP-B3 (JDE)     4000
ERP-B6 (SAP)     3500
ERP-B2 (JDE)     3000
ERP-B1 (JDE)     2025
ERP-B5 (JDE)     1500 

region
R4    11000
R7    11000
R8     5003
R3     4000
R6     3500
R2     3000
R1     2025
R5     1500 

currency
USD    17959
EUR     7283
BRL     2968
GBP     2447
CAD     1830
MXN     1596
MMK     1077
ARS      922
RSD      807
TRY      731
RUB      606
CZK      581
INR      521
PYG      440
CLP      281
THB      270
SAR      189
EGP      179
SEK      105
PLN      101
DKK       63
CHF       46
HKD       20
RCN        3
PEN        3 

document_type
PV    21138
RE    18824
CA      253
Z6      169
PD      130
Z7       96
PM       88
PL       77
PR       58
AB       47
ZG       46
PP       34
ZE       31
ZZ       20
R1       11
KG        3
R3        3 

po_type
F     19503
OK     5374
OP     4524
OC     4257
       3620
OA     3338
FS      304
O3       58
FT       22
OH       18
FD       10 

trans

In [29]:
for c in ["posting_date", "invoice_date", "due_date", "paid_date"]:
    print(c, b[c].map(shape_of).value_counts().to_dict())
    dt = pd.to_datetime(b[c], errors="coerce")
    print("  unparsable (non-blank):", (dt.isna() & b[c].ne("")).sum(), " range:", dt.min(), "to", dt.max())
bd = b.assign(pd_=pd.to_datetime(b.posting_date, errors="coerce"), id_=pd.to_datetime(b.invoice_date, errors="coerce"))
print(bd.pd_.dt.to_period("M").value_counts().sort_index().to_string())
print(bd.id_.dt.year.value_counts().sort_index().to_string())
print("invoice_date after posting_date:", (bd.id_ > bd.pd_).sum())

posting_date {'9999-99-99': 41028}
  unparsable (non-blank): 0  range: 2023-12-31 00:00:00 to 2024-12-29 00:00:00
invoice_date {'9999-99-99': 41028}
  unparsable (non-blank): 0  range: 2018-07-23 00:00:00 to 2027-02-12 00:00:00
due_date {'9999-99-99': 41028}
  unparsable (non-blank): 0  range: 2020-07-30 00:00:00 to 2027-03-14 00:00:00
paid_date {'9999-99-99': 30706, '': 10322}
  unparsable (non-blank): 0  range: 2024-01-02 00:00:00 to 2025-03-30 00:00:00
pd_
2023-12      13
2024-01    3649
2024-02    3205
2024-03    4081
2024-04    3212
2024-05    3380
2024-06    3050
2024-07    3678
2024-08    3487
2024-09    3048
2024-10    3434
2024-11    3142
2024-12    3649
Freq: M
id_
2018        2
2020        1
2021       21
2022       52
2023     1641
2024    39309
2027        2
invoice_date after posting_date: 233


In [31]:
raw_amt = b["line_amount"].astype("string").str.strip().str.replace(",", "", regex=False)
b["amt"] = pd.to_numeric(raw_amt, errors="coerce")

print("missing/unparseable:", b["amt"].isna().sum())
print(b["amt"].describe(), "\nnegative:", (b["amt"] < 0).sum(), " zero:", (b["amt"] == 0).sum())

b["abs_amt"] = b["amt"].abs().round(2)
g = b.groupby(["invoice_number", "vendor_id", "abs_amt"]).amt.agg(["count", "sum"])
pairs = g[(g["count"] >= 2) & (g["sum"].abs() < 0.01)]
print("reversal groups (net to zero):", len(pairs))
print(b.groupby("document_type").amt.agg(["size", lambda s: (s < 0).mean(), "sum"]).round(2).sort_values("size", ascending=False))

missing/unparseable: 3
count         41025.0
mean      9103.725939
std      89323.361513
min       -3406645.05
25%             131.5
50%            721.15
75%           5108.93
max        6924157.27
Name: amt, dtype: Float64 
negative: 833  zero: 918
reversal groups (net to zero): 308
                size  <lambda_0>           sum
document_type                                 
PV             21138        0.01  267581694.37
RE             18824        0.02  107760606.31
CA               253         0.0    1288657.68
Z6               169        0.12       8960.57
PD               130        0.92   -3904675.72
Z7                96        0.02       1148.82
PM                88         0.0     319958.48
PL                77        0.18      297472.9
PR                58         0.0      78652.47
AB                47         1.0    -260792.98
ZG                46         0.0     243090.06
PP                34         0.5           0.0
ZE                31        0.03      63404.32
ZZ       

In [32]:
b["rate"] = pd.to_numeric(b.currency_rate, errors="coerce")
print(pd.crosstab(b.currency, b.rate.isna(), margins=True).sort_values("All", ascending=False))
print("USD lines with a rate:", (b[b.currency == "USD"].rate.notna()).sum())
print("rate==1 on non-USD:", ((b.currency != "USD") & (b.rate == 1)).sum())

fx["rate"] = fx.units_per_usd.astype(float)
print(fx.month.min(), fx.month.max(), fx.currency.nunique())
cover = fx.pivot(index="currency", columns="month", values="rate")
print(cover.isna().sum(axis=1).sort_values(ascending=False).head(12))
print("currencies in B not in FX file:", sorted(set(b.currency) - set(fx.currency)))

rate      False   True    All
currency                     
All       21005  20023  41028
USD        1279  16680  17959
EUR        7183    100   7283
BRL        2966      2   2968
GBP        2423     24   2447
CAD           0   1830   1830
MXN        1586     10   1596
MMK           0   1077   1077
ARS         922      0    922
RSD         807      0    807
TRY         731      0    731
RUB         606      0    606
CZK         578      3    581
INR         521      0    521
PYG         440      0    440
CLP         281      0    281
THB           0    270    270
SAR         189      0    189
EGP         179      0    179
SEK         105      0    105
PLN         101      0    101
DKK          63      0     63
CHF          42      4     46
HKD           0     20     20
RCN           0      3      3
PEN           3      0      3
USD lines with a rate: 1279
rate==1 on non-USD: 0
2022-12 2025-12 30
currency
AED    21
PLN    15
PEN    14
SEK    12
DKK    12
CHF    11
SAR    11
TRY    11
EG

In [33]:
b["inv_month"] = b.invoice_date.str[:7]
b["post_month"] = b.posting_date.str[:7]
for mcol in ["inv_month", "post_month"]:
    j = b[b.rate.notna() & (b.currency != "USD")].merge(
        fx[["currency", "month", "rate"]].rename(columns={"rate": "fx", "month": mcol}), on=["currency", mcol], how="left")
    r = (j.rate / j.fx)
    print(mcol, "matched:", j.fx.notna().mean().round(3), " rate/fx median:", r.median().round(3),
          " within 5%:", ((r - 1).abs() < 0.05).mean().round(3))

inv_month matched: 0.999  rate/fx median: 1.0  within 5%: 0.986
post_month matched: 1.0  rate/fx median: 1.0  within 5%: 0.991


In [34]:
b["desc"] = b.line_description.str.strip()
print("blank descriptions:", (b.desc == "").mean().round(3))
print(pd.crosstab(b.region, b.desc == "", normalize="index").round(2))
print(b[b.desc != ""].desc.sample(40, random_state=0).to_string(index=False))
print(b.desc.value_counts().head(25))

blank descriptions: 0.407
desc    False  True 
region              
R1       0.04   0.96
R2       0.56   0.44
R3       0.55   0.45
R4       0.90   0.10
R5       0.50   0.50
R6       0.00   1.00
R7       0.86   0.14
R8       0.04   0.96
   NATURAL LT RETRO 6X12 880328N2 - NEW STYLE AB TOP FRAME-AN2
                                100% GLYCOL FOR COOLING SYSTEM
                                                    0054718982
                                                    0054451667
                                                    0054574792
                                                    0054440587
      BUD LT CHELADA 25 880141 P2 - NEW STYLE AB TOP FRAME-AP2
                      CBS ALU 1475.5x0.290(473/400) SPRSLK DDP
                                                    0054440870
                BUSHING,FLANGE,0.502" ID - 0.6905" OD,3/4" OAL
                 BIG K LEMON LIME N2 - STANDARD STD PKGING-ST2
                                                    0054679250
        

In [35]:
print(b.groupby(["gl_account", "gl_account_description"]).agg(n=("amt", "size"), usd_like=("amt", "sum"))
        .sort_values("n", ascending=False).head(40))
print(b.groupby("vendor_id").size().describe())
print("vendors in more than one region:", (b.groupby("vendor_id").region.nunique() > 1).sum())
print("blank material_number by region:\n", pd.crosstab(b.region, b.material_number == "", normalize="index").round(2))

                                                 n     usd_like
gl_account gl_account_description                              
GL40251    AP - Other                        11222   22366139.6
GL40125    PEUS                               2974  22095286.54
GL40035    PEUS                               2460  56165869.04
GL40255    AP - Metal                         2160  52112196.74
GL40230    TRADE PAYABLE NATIONAL             1973   2976552.51
GL40279    AP - Freight                       1959   1459769.99
GL40069    PECA                               1635    912244.84
GL40170    P106                               1435   6458967.27
GL40169    P105                               1275   5562920.51
GL40125    PEMU                               1251  45728367.35
GL40170    P110                               1023   5329934.53
GL40007    VAPL                                841    903592.82
           VAPI                                833  12471471.19
GL40200    PEUS                         

In [36]:
s = seed.merge(b, on="line_id", how="left")
print(seed.include.value_counts(), "\n", seed.exclusion_reason.value_counts(), "\n")
print(seed.category_l1.value_counts(dropna=False))

ex = s[s.include == "0"]
for c in ["document_type", "gl_account_description", "po_type", "supplier_type"]:
    print("\nexcluded by", c); print(ex[c].value_counts().head(10).to_string())
    print("share of each value that is excluded (seed):")
    print((s.groupby(c).include.apply(lambda x: (x == "0").mean())).loc[ex[c].value_counts().head(10).index].round(2).to_string())
print(ex.desc.value_counts().head(15) if "desc" in ex else ex.line_description.value_counts().head(15))

include
1    2550
0     450
Name: count, dtype: int64 
 exclusion_reason
                   2550
NON_PROCUREMENT     450
Name: count, dtype: int64 

category_l1
Direct Materials         1400
Indirect Materials        702
                          450
Logistics                 131
Facilities                119
Packaging Materials        67
Professional Services      62
IT & Telecom               46
HR & Employee              15
Utilities                   8
Name: count, dtype: int64

excluded by document_type
document_type
PV    245
RE    198
PL      3
PD      1
ZG      1
ZE      1
CA      1
share of each value that is excluded (seed):
document_type
PV    0.14
RE    0.17
PL    0.25
PD    1.00
ZG    0.14
ZE    0.33
CA    1.00

excluded by gl_account_description
gl_account_description
IC06           107
IC Payables    104
I/C Payable     97
PEUS            74
IC05            29
PECA            10
IC67             7
IC37             5
IC22             5
IC23             4
share of each val

In [37]:
inc = s[s.include == "1"]
print(inc.groupby(["category_l1", "category_l2", "category_l3"]).size().to_string())
# same description, different label?
dd = inc[inc.line_description.str.strip() != ""].groupby("line_description").category_l3.nunique()
print("descriptions with conflicting L3:", (dd > 1).sum(), "of", len(dd))
# which taxonomy leaves never appear in the seed?
leaves = set(taxB.category_l3); seen = set(inc.category_l3)
print("leaves with no seed examples:", sorted(leaves - seen))

category_l1            category_l2               category_l3                    
Direct Materials       Aluminium                 Can Body Stock (coil)              878
                                                 Can End Stock (coil)                78
                                                 Ingot & Metal Premium              311
                                                 Tab Stock (coil)                    75
                       Coatings & Inks           Inks & Varnishes                    26
                                                 Internal Coatings & Compounds       11
                       Ends & Closures           Can Ends                            21
Facilities             Environmental             Waste Disposal & Recycling          11
                       Facility Services         Cleaning                            28
                                                 Equipment & Site Rental             12
                                       

In [38]:
for name, tx in [("A", taxA), ("B", taxB)]:
    print(name, tx.shape, "L1:", tx.category_l1.nunique(), "L2:", tx.category_l2.nunique(), "L3:", tx.category_l3.nunique())
    print("L3 names not unique:", tx.category_l3.duplicated().sum(), 
          " L2==L3:", (tx.category_l2 == tx.category_l3).sum())
# B's labels must exist in the tree
print("B seed labels not in tree:",
      set(seed.category_l3[seed.category_l3 != ""]) - set(taxB.category_l3))
print("A labels not in tree:", set(labA.category_l3) - set(taxA.category_l3))

A (40, 3) L1: 5 L2: 24 L3: 40
L3 names not unique: 0  L2==L3: 21
B (40, 3) L1: 9 L2: 25 L3: 40
L3 names not unique: 0  L2==L3: 1
B seed labels not in tree: set()
A labels not in tree: set()
